# Process Expression Notebook

imports + paths + file check

In [5]:
from pathlib import Path
import os

# set project root = parent of /notebooks
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("CWD set to:", Path.cwd())

CWD set to: c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project


In [6]:
from pathlib import Path
import re
import pandas as pd

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

expr_path = RAW / "OmicsExpressionProteinCodingGenesTPMLogp1.csv"
expr_path.exists(), expr_path

(True, WindowsPath('data/raw/OmicsExpressionProteinCodingGenesTPMLogp1.csv'))

helper to clean gene column names

In [7]:
def clean_gene_cols(cols):
    """
    Convert: 'TSPAN6 (7105)' -> 'TSPAN6'
    Keep uniqueness by appending '__2', '__3', etc. if needed.
    """
    cleaned = []
    seen = {}
    pat = re.compile(r"^(.*)\s+\(\d+\)$")

    for c in cols:
        m = pat.match(c)
        base = (m.group(1) if m else c).strip()
        seen[base] = seen.get(base, 0) + 1
        cleaned.append(base if seen[base] == 1 else f"{base}__{seen[base]}")
    return cleaned

read + inspect

In [8]:
df = pd.read_csv(expr_path)
df.shape, df.columns[:5]

((1517, 19194),
 Index(['Unnamed: 0', 'TSPAN6 (7105)', 'TNMD (64102)', 'DPM1 (8813)',
        'SCYL3 (57147)'],
       dtype='str'))

standardize ID + clean gene cols

In [9]:
id_col = df.columns[0]  # usually 'Unnamed: 0'
df = df.rename(columns={id_col: "depmap_id"})

gene_cols = [c for c in df.columns if c != "depmap_id"]
cleaned = clean_gene_cols(gene_cols)

feature_map = pd.DataFrame({"original": gene_cols, "feature": cleaned})
feature_map.head()

,original,feature
0,TSPAN6 (7105),TSPAN6
1,TNMD (64102),TNMD
2,DPM1 (8813),DPM1
3,SCYL3 (57147),SCYL3
4,C1orf112 (55732),C1orf112


save processed outputs

In [10]:
feature_map.to_csv(OUT / "expression_feature_map.csv", index=False)

df.columns = ["depmap_id"] + cleaned
df.to_parquet(OUT / "expression_tpm_logp1.parquet", index=False)

print("Wrote:", OUT / "expression_tpm_logp1.parquet")
print("Wrote:", OUT / "expression_feature_map.csv")
print("Rows:", f"{df.shape[0]:,}", "| Features:", f"{df.shape[1]-1:,}")
df.head()

Wrote: data\processed\expression_tpm_logp1.parquet
Wrote: data\processed\expression_feature_map.csv
Rows: 1,517 | Features: 19,193


,depmap_id,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,...,H3C2,H3C3,AC098582.1,DUS4L-BCAP29,C8orf44-SGK3,ELOA3B,NPBWR1,ELOA3D,ELOA3,CDR1
0,ACH-001113,4.331992,0.000000,7.364660,2.792855,4.471187,0.028569,1.226509,3.044394,6.500005,...,2.689299,0.189034,0.201634,2.130931,0.555816,0.0,0.275007,0.0,0.0,0.000000
1,ACH-001289,4.567424,0.584963,7.106641,2.543496,3.504620,0.000000,0.189034,3.813525,4.221877,...,1.286881,1.049631,0.321928,1.464668,0.632268,0.0,0.014355,0.0,0.0,0.000000
2,ACH-001339,3.150560,0.000000,7.379118,2.333424,4.228049,0.056584,1.310340,6.687201,3.682573,...,0.594549,1.097611,0.831877,2.946731,0.475085,0.0,0.084064,0.0,0.0,0.042644
3,ACH-001538,5.085340,0.000000,7.154211,2.545968,3.084064,0.000000,5.868390,6.165309,4.489928,...,0.214125,0.632268,0.298658,1.641546,0.443607,0.0,0.028569,0.0,0.0,0.000000
4,ACH-000242,6.729417,0.000000,6.537917,2.456806,3.867896,0.799087,7.208478,5.570159,7.127117,...,1.117695,2.358959,0.084064,1.910733,0.000000,0.0,0.464668,0.0,0.0,0.000000
